[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_8_deep_sarsa_complete.ipynb)

<div style="text-align:center">
    <h1>
        Deep SARSA
    </h1>
</div>

<br><br>

<div style="text-align:center">

In this notebook, we extend the SARSA algorithm to use function approximators (Neural Networks). The resulting algorithm is known as Deep SARSA.
</div>


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_stats, seed_everything, plot_cost_to_go, plot_max_q


## Import the necessary software libraries:

In [ ]:
# random: drives epsilon-greedy exploration and replay-buffer sampling.
import random
# copy: used to deep-copy the Q-network into a separate target network.
import copy
# gym: the MountainCar environment and wrapper base class.
import gym
# torch: the deep-learning library that holds tensors and computes gradients.
import torch
# functional API: gives us mse_loss for the regression target.
import torch.nn.functional as F
import matplotlib.pyplot as plt
# nn: layers (Linear, ReLU) and the Sequential container for the network.
from torch import nn as nn
# AdamW: the optimizer that updates the network weights from the gradients.
from torch.optim import AdamW
# tqdm: progress bar over training episodes.
from tqdm import tqdm

## Create and prepare the environment

### Create the environment

In [ ]:
# Create the continuous-state MountainCar task.
env = gym.make('MountainCar-v0')
# Seed everything for reproducibility.
seed_everything(env)

In [ ]:
# How many numbers describe a state (here 2: position and velocity)...
state_dims = env.observation_space.shape[0]
# ...and how many discrete actions the agent can take (here 3).
num_actions = env.action_space.n
print(f"MountainCar env: State dimensions: {state_dims}, Number of actions: {num_actions}")

### Prepare the environment to work with PyTorch

### Getting the environment ready for PyTorch

A neural network speaks **tensors**, but gym hands back NumPy arrays. The wrapper below converts every observation, reward, and done-flag into a PyTorch tensor and adds a leading **batch dimension** (shape `1 x state_dims`), because PyTorch layers always expect a batch of inputs. After this, data flows straight from the environment into the network with no manual conversion.

In [ ]:
# A Wrapper that converts the env's NumPy arrays into PyTorch tensors,
# so the data flows straight into the neural network. It also adds a batch
# dimension (unsqueeze) because PyTorch layers expect a batch of inputs.
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        gym.Wrapper.__init__(self, env)

    def reset(self):
        # Reset the underlying env, then turn the state into a 1 x state_dims float tensor.
        obs = self.env.reset()
        return torch.from_numpy(obs).unsqueeze(dim=0).float()

    def step(self, action):
        # The action arrives as a 1x1 tensor; .item() extracts the plain int gym expects.
        action = action.item()
        next_state, reward, done, info = self.env.step(action)
        # Wrap the next state as a 1 x state_dims float tensor (with batch dim).
        next_state = torch.from_numpy(next_state).unsqueeze(dim=0).float()
        # Wrap reward and done as 1x1 tensors so they batch cleanly later.
        reward = torch.tensor(reward).view(1, -1).float()
        done = torch.tensor(done).view(1, -1)
        return next_state, reward, done, info

In [ ]:
# Wrap the environment so every observation it returns is already a tensor.
env = PreprocessEnv(env)

In [ ]:
# Quick sanity check: take one random step and inspect the tensor shapes/values.
state = env.reset()
action = torch.tensor(0)
next_state, reward, done, _ = env.step(action)
print(f"Sample state: {state}")
print(f"Next state: {next_state}, Reward: {reward}, Done: {done}")

## Create the Q-Network and policy

<br><br>

### Create the Q-Network: $\hat q(s,a| \theta)$

### Approximating Q-values with a neural network

A table cannot cover MountainCar's infinitely many continuous states. Instead we use a **function approximator**: a small neural network that takes the raw state `(position, velocity)` and outputs an estimated Q-value for **each** action. Two ideas make this work well:

1. **The Q-network `q(s,a|theta)`** - the weights `theta` are what we learn. Because it is a smooth function, states that are close together get similar Q-values automatically (the *generalization* that the table lacked).
2. **A separate target network `q(s,a|theta_targ)`** - a frozen copy used to compute the learning target. If we used the same fast-changing network for both the prediction and the target, the target would shift every update and training would chase a moving goal. Refreshing the target only occasionally keeps it stable.

The cells below build the network, its frozen target copy, and an epsilon-greedy policy that reads action values straight from the network.

In [ ]:
# The Q-network: a small multilayer perceptron that maps a state to one
# Q-value PER action. Input = state_dims (2), output = num_actions (3).
q_network = nn.Sequential(
    # First layer: 2 inputs -> 128 hidden units.
    nn.Linear(state_dims, 128),
    # Non-linearity so the network can fit a curved value surface.
    nn.ReLU(),
    # Second hidden layer: 128 -> 64.
    nn.Linear(128, 64),
    nn.ReLU(),
    # Output layer: 64 -> 3 Q-values (one per action). No activation: Q-values are unbounded.
    nn.Linear(64, num_actions))

### Create the target Q-Network: $\hat q(s, a|\theta_{targ})$

In [ ]:
# The target network is a FROZEN copy of the Q-network used to compute the
# learning target. Keeping it fixed for a while stabilises training (the target
# stops chasing its own tail). .eval() puts it in inference mode.
target_q_network = copy.deepcopy(q_network).eval()

### Create the $\epsilon$-greedy policy: $\pi(s)$

In [ ]:
# Epsilon-greedy policy that reads Q-values from the network.
def policy(state, epsilon=0.):
    # With probability epsilon, explore with a random action (as a 1x1 tensor).
    if torch.rand(1) < epsilon:
        return torch.randint(num_actions, (1, 1))
    else:
        # Otherwise run the network; .detach() means 'no gradients needed here'.
        av = q_network(state).detach()
        # Pick the action with the highest Q-value, keeping a 1x1 shape.
        return torch.argmax(av, dim=-1, keepdim=True)

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
# Visualise the UNTRAINED value surface (cost-to-go) before any learning.
plot_cost_to_go(env, q_network, xlabel='Car Position', ylabel='Velocity')

## Create the Experience Replay buffer

<br>
<div style="text-align:center">
    <p>A simple buffer that stores transitions of arbitrary values, adapted from
    <a href="https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html#training">this source.</a></p>
</div>


### Experience Replay: why we don't learn from each step in order

If we trained on transitions exactly as they arrive, consecutive samples would be highly **correlated** (one step looks a lot like the next), which makes gradient descent unstable and prone to forgetting. The **replay buffer** stores many past transitions and lets us draw a **random minibatch** each update. This decorrelates the data and reuses each experience many times, so the network learns more stably and efficiently.

In [ ]:
# Experience Replay buffer: stores past transitions so we can train on
# random minibatches instead of only the latest step. This breaks the strong
# correlation between consecutive steps and makes the gradient updates stabler.
class ReplayMemory:

    def __init__(self, capacity=1000000):
        self.capacity = capacity
        self.memory = []
        # position is a write cursor for the circular buffer.
        self.position = 0

    def insert(self, transition):
        # Grow the list until it reaches capacity...
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        # ...then overwrite the oldest slot (ring buffer behaviour).
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        # Only sample once we have enough data (see can_sample below).
        assert self.can_sample(batch_size)

        # Draw a random minibatch of transitions.
        batch = random.sample(self.memory, batch_size)
        # Transpose list-of-transitions into transition-of-lists (states, actions, ...).
        batch = zip(*batch)
        # Concatenate each field into one stacked tensor for the whole batch.
        return [torch.cat(items) for items in batch]

    def can_sample(self, batch_size):
        # Heuristic: wait until the buffer holds 10x the batch size before sampling.
        return len(self.memory) >= batch_size * 10

    def __len__(self):
        return len(self.memory)

## Implement the algorithm

</br></br>

### Deep SARSA: SARSA with a neural network in place of the table

We now replace the Q-table with the neural network `q_network`. Read the training loop with the tabular SARSA update in mind - the structure is the same, only the pieces are 'deep':

- **The estimate `Q(s,a)`** comes from a forward pass through the network instead of a table lookup. `gather(1, action_b)` selects, for each example in the batch, the Q-value of the action that was actually taken.
- **The SARSA target** is `r + gamma * Q(s', a')`. The `a'` is chosen by the *same* epsilon-greedy policy (that on-policy choice is what makes this SARSA, not Q-Learning, which would use the max).
- **The target network** supplies `Q(s', a')`. Because it is a frozen copy refreshed only every 10 episodes, the target moves slowly and training does not oscillate.
- **`~done_b`** zeroes the future term on the last step of an episode: there is no `s'` to bootstrap from once the episode is over.
- **Learning** is a regression: we minimise the mean-squared error between the network's `Q(s,a)` and the target, then take one `AdamW` step (`zero_grad -> backward -> step`).

This is the bridge from tabular RL to deep RL: gradient descent on a network replaces the simple in-place table update.

In [ ]:
# Deep SARSA: the SARSA update rule, but Q is now a neural network.
def deep_sarsa(q_network, policy, episodes, alpha=0.001,
               batch_size=32, gamma=0.99, epsilon=0.05):
    # AdamW optimizer adjusts the network weights to reduce the loss.
    optim = AdamW(q_network.parameters(), lr=alpha)
    # The replay buffer that feeds minibatches to the optimizer.
    memory = ReplayMemory()
    # Track the loss and the return for plotting later.
    stats = {'MSE Loss': [], 'Returns': []}

    for episode in tqdm(range(1, episodes + 1)):
        state = env.reset()
        done = False
        ep_return = 0
        while not done:
            # SARSA is on-policy: choose the action from the epsilon-greedy policy.
            action = policy(state, epsilon)
            next_state, reward, done, _ = env.step(action)
            # Store this transition for later replay.
            memory.insert([state, action, reward, done, next_state])

            if memory.can_sample(batch_size):
                # Pull a random minibatch of past transitions.
                state_b, action_b, reward_b, done_b, next_state_b = memory.sample(batch_size)
                # Q(s,a) for the actions actually taken: gather picks one column per row.
                qsa_b = q_network(state_b).gather(1, action_b)
                # SARSA target uses the action the policy WOULD take next (a'), not the max.
                next_action_b = policy(next_state_b, epsilon)
                # Q(s', a') from the FROZEN target network for stability.
                next_qsa_b = target_q_network(next_state_b).gather(1, next_action_b)
                # TD target: r + gamma * Q(s', a'); the ~done_b zeroes it out at episode end.
                target_b = reward_b + ~done_b * gamma * next_qsa_b
                # Regression loss: push the network's Q(s,a) toward that target.
                loss = F.mse_loss(qsa_b, target_b)
                # Clear old gradients before backprop.
                q_network.zero_grad()
                # Backpropagate the loss to get gradients for every weight.
                loss.backward()
                # Take one optimizer step to update the weights.
                optim.step()

                stats['MSE Loss'].append(loss.item())

            state = next_state
            ep_return += reward.item()

        stats['Returns'].append(ep_return)

        # Every 10 episodes, copy the live weights into the target network.
        if episode % 10 == 0:
            target_q_network.load_state_dict(q_network.state_dict())

    return stats

In [ ]:
# Train Deep SARSA for 2500 episodes with a small exploration rate.
stats = deep_sarsa(q_network, policy, 2500, epsilon=0.01)

## Show results

### Plot execution stats

In [ ]:
# Plot the MSE loss and the episode returns over training.
plot_stats(stats)

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
# Cost-to-go surface AFTER training: should now form the classic MountainCar valley.
plot_cost_to_go(env, q_network, xlabel='Car Position', ylabel='Velocity')

### Show resulting policy: $\pi(s)$

In [ ]:
# Show which action the trained network prefers across the state space.
plot_max_q(env, q_network, xlabel='Car Position', ylabel='Velocity',
           action_labels=['Back', 'Do nothing', 'Forward'])

### Test the resulting agent

In [ ]:
# Run the greedy learned policy for 2 episodes to watch it solve the task.
test_agent(env, policy, episodes=2)

## Resources

[[1] Deep Reinforcement Learning with Experience Replay Based on SARSA](https://www.researchgate.net/publication/313803199_Deep_reinforcement_learning_with_experience_replay_based_on_SARSA)